# Training: Six-Class Age-Group Image Classification

**Target mapping:** `ImageFolder` assigns class indices alphabetically. This notebook verifies the six age-group classes and uses the same mapping for training, validation, and testing.


#### Import the required libraries:

In [152]:
import copy
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler   # CHANGED: added WeightedRandomSampler
from torchvision import datasets, transforms, models

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,auc,
                             confusion_matrix, roc_auc_score, roc_curve, classification_report)
import plotly.express as px
import plotly.graph_objects as go

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce GTX 1650 Ti


### Load the dataset

`ImageFolder` treats each class subfolder as a target label and assigns indices in alphabetical order. This notebook expects these six age-group folders: `18-25`, `26-35`, `36-50`, `50+`, `not_applicable`, and `under_18`.


In [111]:
# Set the project directory.
PROJECT_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final")

# Set the data and model directories.
DATA_DIR = PROJECT_DIR / "data"
TASK = "age"

SPLIT_DIR = DATA_DIR / "split_age"

# Save trained models in the existing project-level models directory.
MODEL_DIR = PROJECT_DIR / "models" / "age_group"
MODEL_DIR.mkdir(parents=True, exist_ok=True)


CHARTS_DIR = PROJECT_DIR / "charts" / "age_group"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)


# Define the training, validation, and test directories.
TRAIN_DIR = SPLIT_DIR / "train"
VAL_DIR = SPLIT_DIR / "val"
TEST_DIR = SPLIT_DIR / "test"

# Check that all split directories exist.
split_dirs = [TRAIN_DIR, VAL_DIR, TEST_DIR]
missing_splits = [
    str(folder) for folder in split_dirs
    if not folder.is_dir()
]

if missing_splits:
    raise FileNotFoundError(
        "The following split directories are missing: "
        + ", ".join(missing_splits)
        + ". Run 03_data_split_train_val_test_split.ipynb first."
    )

# Class folders expected in every split (alphabetical order = ImageFolder index order)
EXPECTED_CLASSES = ["18-25", "26-35", "36-50", "50+", "not_applicable", "under_18"]
# EXPECTED_CLASSES = ["18-25", "26-35", "36-50", "50+", "under_18"]


for split_dir in split_dirs:
    found_classes = sorted(folder.name for folder in split_dir.iterdir() if folder.is_dir())
    if found_classes != EXPECTED_CLASSES:
        raise FileNotFoundError(
            f"Class folders in {split_dir} are {found_classes}, expected {EXPECTED_CLASSES}"
        )

# Set training parameters.
BATCH_SIZE = 32  # Reduce to 16 if a CUDA out-of-memory error occurs.
PRETRAINED = True  # Set to False to train without pretrained ImageNet weights.
SEED = 42

# Set random seeds for reproducibility.
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Data directory:", DATA_DIR)
print("Split directory:", SPLIT_DIR)
print("Model save directory:", MODEL_DIR)
print("Charts save directory:", CHARTS_DIR)
print("Training, validation, and test folders are ready.")

Data directory: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data
Split directory: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\split_age
Model save directory: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\age_group
Charts save directory: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\charts\age_group
Training, validation, and test folders are ready.


# Data Preprocessing:

### Transform Compose:

Red channel: mean = 0.485, std = 0.229

Green channel: mean = 0.456, std = 0.224

Blue channel: mean = 0.406, std = 0.225

### Train:

In [112]:
train_transform = transforms.Compose([
                                        transforms.RandomHorizontalFlip(p=0.5),
                                        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
                                        transforms.RandomRotation(degrees=10),
                                        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
                                        transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                             std=[0.229, 0.224, 0.225]),
                                        transforms.RandomErasing(p=0.25),
                                        
                                    ])

### Test/Val:

In [113]:
val_test_transform = transforms.Compose([
                                transforms.Resize((224, 224)),
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                     std=[0.229, 0.224, 0.225])
                                        ])

In [114]:
# Create datasets for training, training-set evaluation, validation, and testing.
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
train_eval_dataset = datasets.ImageFolder(TRAIN_DIR, transform=val_test_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_test_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_test_transform)

# Confirm that every split uses the same target-to-index mapping.
expected_class_to_idx = {"18-25": 0, "26-35": 1, "36-50": 2, "50+": 3, "not_applicable": 4, "under_18": 5}
# expected_class_to_idx = {"18-25": 0, "26-35": 1, "36-50": 2, "50+": 3, "under_18": 4}
def normalize_class_mapping(class_to_idx):
    return {name.lower(): index for name, index in class_to_idx.items()}

train_mapping = normalize_class_mapping(train_dataset.class_to_idx)
val_mapping = normalize_class_mapping(val_dataset.class_to_idx)
test_mapping = normalize_class_mapping(test_dataset.class_to_idx)

for split_name, mapping in [
    ("training", train_mapping),
    ("validation", val_mapping),
    ("test", test_mapping)
]:
    if mapping != expected_class_to_idx:
        raise ValueError(
            f"Incorrect target mapping in the {split_name} split: {mapping}. "
            "Check the six age-group folder names and their indices."
        )

class_names = train_dataset.classes
num_classes = len(class_names)

if num_classes != 6:
    raise ValueError(f"Expected exactly six classes, but found: {class_names}")

# Check all classes are represented in each split (important for reliable metrics).
for split_name, dataset in [
    ("training", train_dataset),
    ("training evaluation", train_eval_dataset),
    ("validation", val_dataset),
    ("test", test_dataset)
]:
    split_counts = np.bincount(dataset.targets, minlength=num_classes)
    missing = [class_names[i] for i, count in enumerate(split_counts) if count == 0]
    if missing:
        raise ValueError(
            f"The {split_name} split has no images for these classes: {missing}"
        )

print("Training class mapping:", train_dataset.class_to_idx)
print("Validation class mapping:", val_dataset.class_to_idx)
print("Test class mapping:", test_dataset.class_to_idx)
print("Target mapping verified:", expected_class_to_idx)

image, label = train_dataset[0]
print("Example target index:", label, "->", class_names[label])
print("Image type:", type(image))
print("Image shape:", image.shape)


Training class mapping: {'18-25': 0, '26-35': 1, '36-50': 2, '50+': 3, 'not_applicable': 4, 'under_18': 5}
Validation class mapping: {'18-25': 0, '26-35': 1, '36-50': 2, '50+': 3, 'not_applicable': 4, 'under_18': 5}
Test class mapping: {'18-25': 0, '26-35': 1, '36-50': 2, '50+': 3, 'not_applicable': 4, 'under_18': 5}
Target mapping verified: {'18-25': 0, '26-35': 1, '36-50': 2, '50+': 3, 'not_applicable': 4, 'under_18': 5}
Example target index: 0 -> 18-25
Image type: <class 'torch.Tensor'>
Image shape: torch.Size([3, 224, 224])


## Class counts and sampler (CHANGED)

The train set is balanced with a `WeightedRandomSampler` using **square-root inverse frequency**, which raises the share of minority classes without extreme repetition. Because the sampler already corrects the imbalance, the loss function below is **not** class-weighted. Validation and test sets stay in their natural distribution.

In [115]:
counts = np.bincount(train_dataset.targets, minlength=num_classes)
if np.any(counts == 0):
    raise ValueError("A class has no training images.")

for name, count in zip(class_names, counts):
    print(f"{name:15s} images = {count:5d}")

# CHANGED: sqrt inverse-frequency sampling weights
class_sample_weight = (1.0 / counts) ** 0.5
sample_weights = class_sample_weight[np.array(train_dataset.targets)]
sampler = WeightedRandomSampler(
    weights = torch.as_tensor(sample_weights, dtype=torch.double),  #type: ignore
    num_samples=len(train_dataset),
    replacement=True
)

effective_share = class_sample_weight * counts
effective_share = effective_share / effective_share.sum()
print("\nApprox. share of each class per epoch after sampling:")
for name, share in zip(class_names, effective_share):
    print(f"{name:15s} {share:.3f}")

18-25           images =   254
26-35           images =  1089
36-50           images =   223
50+             images =   102
not_applicable  images =  1844
under_18        images =    49

Approx. share of each class per epoch after sampling:
18-25           0.129
26-35           0.266
36-50           0.121
50+             0.082
not_applicable  0.347
under_18        0.056


# Data Loader:

In [116]:
# CHANGED: train_loader uses the sampler (shuffle must be False/omitted when a sampler is given).
train_loader      = DataLoader(train_dataset,      batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
train_eval_loader = DataLoader(train_eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
val_loader        = DataLoader(val_dataset,        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader       = DataLoader(test_dataset,       batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 112
Val batches: 16
Test batches: 32


In [117]:
images, labels = next(iter(train_loader))
print("Batch of images shape:", images.shape)
print("Batch of labels shape:", labels.shape)

Batch of images shape: torch.Size([32, 3, 224, 224])
Batch of labels shape: torch.Size([32])


### Class weights

Class weights are calculated from the training split only. They reduce the influence of class imbalance by assigning larger loss weights to classes with fewer training images.


In [118]:
# # Count training images in each class (same order as class_names).
# counts = np.bincount(train_dataset.targets, minlength=num_classes)

# # Stop if any class has no training images.
# if np.any(counts == 0):
#     missing_classes = [
#         class_names[i] for i, count in enumerate(counts) if count == 0
#     ]
#     raise ValueError(f"Classes with no training images: {missing_classes}")

# # Inverse-frequency weights: minority classes receive larger weights.
# weights = len(train_dataset) / (num_classes * counts)

# # CrossEntropyLoss expects float weights on the same device as model outputs.
# class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

# for name, count, weight in zip(class_names, counts, weights):
#     print(f"{name:15s} images = {count:5d}   weight = {weight:.3f}")


# Model Building

Models are created on the CPU and moved to the GPU only for training to reduce GPU memory usage.

### Model -1 - CNN Model:

In [119]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        def block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=2)
            )

        self.block1 = block(3, 32)
        self.block2 = block(32, 64)
        self.block3 = block(64, 128)
        self.block4 = block(128, 256)
        self.block5 = block(256, 512)

        self.dropout_conv = nn.Dropout2d(0.1)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(512, 256)
        self.dropout_fc = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.dropout_conv(x)

        x = self.gap(x)
        x = torch.flatten(x, start_dim=1)

        x = torch.relu(self.fc1(x))
        x = self.dropout_fc(x)
        x = self.fc2(x)

        return x


num_classes = len(class_names)
cnn_model = CNN(num_classes).to(device)

print(sum(p.numel() for p in cnn_model.parameters()), "parameters")
print(cnn_model)

1703430 parameters
CNN(
  (block1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block4): Sequential(
    (0

### Model - 2 - Transfer Learning (ResNet 50):

In [120]:
PRETRAINED = True
weights_name = models.ResNet50_Weights.IMAGENET1K_V1 if PRETRAINED else None
resnet_model = models.resnet50(weights=weights_name)

for param in resnet_model.parameters():
    param.requires_grad = False

resnet_model.fc = nn.Sequential(                                 #type: ignore
                    nn.Linear(resnet_model.fc.in_features, 256),
                    nn.ReLU(inplace=True),
                    nn.Dropout(0.5),
                    nn.Linear(256, num_classes)
)

# CHANGED: unfreeze layer3 AND layer4.
for param in resnet_model.layer3.parameters():
    param.requires_grad = True
for param in resnet_model.layer4.parameters():
    param.requires_grad = True

print(resnet_model.fc)

Sequential(
  (0): Linear(in_features=2048, out_features=256, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=256, out_features=6, bias=True)
)


### Model - 3 - Transfer Learning (VGG16):

In [121]:
weights_name = models.VGG16_Weights.DEFAULT
vgg_model = models.vgg16(weights=weights_name)

for param in vgg_model.parameters():
    param.requires_grad = False

vgg_model.classifier = nn.Sequential(
    nn.Linear(512 * 7 * 7, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)

# CHANGED: unfreeze from features[17:] (last two conv blocks) instead of features[24:].
for param in vgg_model.features[24:].parameters(): #type: ignore
    param.requires_grad = True

print(vgg_model.classifier)

Sequential(
  (0): Linear(in_features=25088, out_features=256, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=256, out_features=6, bias=True)
)


### Model - 4 - Transfer Learning (EfficientNet-B0):

In [122]:
weights_name = models.EfficientNet_B0_Weights.DEFAULT
effnet_model = models.efficientnet_b0(weights=weights_name)

for param in effnet_model.parameters():
    param.requires_grad = False

effnet_model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(effnet_model.classifier[1].in_features, 256), #type: ignore
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)

# CHANGED: unfreeze from features[5:] instead of features[7:].
for param in effnet_model.features[5:].parameters():
    param.requires_grad = True

print(effnet_model.classifier)

Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=1280, out_features=256, bias=True)
  (2): ReLU(inplace=True)
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=256, out_features=6, bias=True)
)


### Model - 5 - Vision Transformer (ViT):

In [123]:
weights_name = models.ViT_B_16_Weights.DEFAULT
vit_model = models.vit_b_16(weights=weights_name)

for param in vit_model.parameters():
    param.requires_grad = False

vit_model.heads = nn.Sequential(
    nn.Linear(vit_model.heads[0].in_features, 256), #type: ignore
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)

# CHANGED: unfreeze the last THREE encoder blocks (was only the last one).
for layer in vit_model.encoder.layers[-3:]:
    for param in layer.parameters():
        param.requires_grad = True

for param in vit_model.encoder.ln.parameters():
    param.requires_grad = True

print(vit_model.heads)

Sequential(
  (0): Linear(in_features=768, out_features=256, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=256, out_features=6, bias=True)
)


# Model Training:

### Training function

- Each epoch performs training followed by validation.
- The learning rate is reduced when validation loss stops improving.
- Training stops early if validation loss does not improve for five consecutive epochs, and the best weights are restored.

In [124]:
def train_model(model, optimizer, criterion, num_epochs, patience=8):
    history = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": [], "val_f1": []}

    best_f1 = -1.0                                    # CHANGED
    best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    best_epoch = 0
    epochs_without_improvement = 0

    # CHANGED: scheduler follows val macro-F1 (maximize)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    for epoch in range(num_epochs):
        # ---- Training ----
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        train_loss = running_loss / total
        history["train_acc"].append(train_acc)
        history["train_loss"].append(train_loss)

        # ---- Validation ----
        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0
        all_p, all_l = [], []                         # CHANGED

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_running_loss += loss.item() * images.size(0)
                preds = outputs.argmax(1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                all_p.extend(preds.cpu().numpy())     # CHANGED
                all_l.extend(labels.cpu().numpy())    # CHANGED

        val_loss = val_running_loss / val_total
        val_acc = val_correct / val_total
        val_f1 = f1_score(all_l, all_p, labels=np.arange(num_classes),
                          average="macro", zero_division=0)   # CHANGED
        history["val_acc"].append(val_acc)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)

        scheduler.step(val_f1)                        # CHANGED

        print(
            f"Epoch {epoch + 1}/{num_epochs}: "
            f"Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, "
            f"Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, Val macro-F1={val_f1:.4f}"
        )

        # ---- Keep best weights (by val macro-F1) + early stopping ----
        if val_f1 > best_f1:                          # CHANGED
            best_f1 = val_f1
            best_epoch = epoch + 1
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    print(f"Best epoch: {best_epoch} (val macro-F1 = {best_f1:.4f})")
    model.load_state_dict(best_weights)
    return history

### CNN Model:

In [125]:
cnn_model = cnn_model.to(device)

# CHANGED: no class weights (sampler handles imbalance), label smoothing added.
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, cnn_model.parameters()),
    lr=1e-5,                                          # CHANGED (was 1e-3)
    weight_decay=1e-4
)

cnn_history = train_model(cnn_model, optimizer, criterion, num_epochs=30)   # CHANGED

cnn_model = cnn_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Epoch 1/30: 100%|██████████| 112/112 [01:24<00:00,  1.33it/s]


Epoch 1/30: Train Loss=1.4414, Val Loss=1.1101, Train Acc=0.5288, Val Acc=0.6673, Val macro-F1=0.2422


Epoch 2/30: 100%|██████████| 112/112 [01:27<00:00,  1.29it/s]


Epoch 2/30: Train Loss=1.3139, Val Loss=1.1021, Train Acc=0.5580, Val Acc=0.6732, Val macro-F1=0.2453


Epoch 3/30: 100%|██████████| 112/112 [01:33<00:00,  1.20it/s]


Epoch 3/30: Train Loss=1.2658, Val Loss=1.0940, Train Acc=0.5805, Val Acc=0.6811, Val macro-F1=0.2493


... (34 outputs left out)

Epoch 21/30: 100%|██████████| 112/112 [01:28<00:00,  1.27it/s]


Epoch 21/30: Train Loss=1.1861, Val Loss=1.0536, Train Acc=0.6046, Val Acc=0.6949, Val macro-F1=0.2779


Epoch 22/30: 100%|██████████| 112/112 [01:36<00:00,  1.16it/s]


Epoch 22/30: Train Loss=1.1744, Val Loss=1.0601, Train Acc=0.6043, Val Acc=0.6929, Val macro-F1=0.2736
Early stopping at epoch 22
Best epoch: 14 (val macro-F1 = 0.2791)


In [126]:
# Save the trained Custom CNN age-group model weights.
save_path = MODEL_DIR / "custom_cnn_age_group.pth"
torch.save(cnn_model.state_dict(), save_path)

print(f"Saved Custom CNN age-group model to: {save_path}")


Saved Custom CNN age-group model to: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\age_group\custom_cnn_age_group.pth


### ResNet 50:

In [127]:
resnet_model = resnet_model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)   # CHANGED
optimizer = optim.AdamW(
    [
        {"params": resnet_model.layer3.parameters(), "lr": 1e-5},   # CHANGED
        {"params": resnet_model.layer4.parameters(), "lr": 1e-5},
        {"params": resnet_model.fc.parameters(),     "lr": 1e-3},
    ],
    weight_decay=0.05
)

resnet_history = train_model(resnet_model, optimizer, criterion, num_epochs=25)   # CHANGED

resnet_model = resnet_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Epoch 1/25: 100%|██████████| 112/112 [01:49<00:00,  1.02it/s]


Epoch 1/25: Train Loss=1.2322, Val Loss=1.0645, Train Acc=0.5835, Val Acc=0.7047, Val macro-F1=0.3944


Epoch 2/25: 100%|██████████| 112/112 [01:52<00:00,  1.01s/it]


Epoch 2/25: Train Loss=1.0914, Val Loss=1.1487, Train Acc=0.6479, Val Acc=0.7224, Val macro-F1=0.3314


Epoch 3/25: 100%|██████████| 112/112 [01:48<00:00,  1.04it/s]


Epoch 3/25: Train Loss=0.9894, Val Loss=1.1284, Train Acc=0.6987, Val Acc=0.6929, Val macro-F1=0.4202


... (12 outputs left out)

Epoch 10/25: 100%|██████████| 112/112 [01:51<00:00,  1.00it/s]


Epoch 10/25: Train Loss=0.7611, Val Loss=1.2749, Train Acc=0.8405, Val Acc=0.6890, Val macro-F1=0.3740


Epoch 11/25: 100%|██████████| 112/112 [01:49<00:00,  1.02it/s]


Epoch 11/25: Train Loss=0.7544, Val Loss=1.2955, Train Acc=0.8402, Val Acc=0.7067, Val macro-F1=0.3752
Early stopping at epoch 11
Best epoch: 3 (val macro-F1 = 0.4202)


In [128]:
# Save the trained ResNet50 age-group model weights.
save_path = MODEL_DIR / "resnet50_age_group.pth"
torch.save(resnet_model.state_dict(), save_path)

print(f"Saved ResNet50 age-group model to: {save_path}")


Saved ResNet50 age-group model to: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\age_group\resnet50_age_group.pth


### VGG16:

In [129]:
vgg_model = vgg_model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)   # CHANGED
optimizer = optim.AdamW(
    [
        {"params": vgg_model.features[24:].parameters(), "lr": 1e-5},    #type: ignore
        {"params": vgg_model.classifier.parameters(),    "lr": 1e-3},
    ],
    weight_decay=0.05
)

vgg_history = train_model(vgg_model, optimizer, criterion, num_epochs=25)   # CHANGED

vgg_model = vgg_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Epoch 1/25: 100%|██████████| 112/112 [01:52<00:00,  1.01s/it]


Epoch 1/25: Train Loss=1.1984, Val Loss=1.2062, Train Acc=0.6172, Val Acc=0.6909, Val macro-F1=0.3298


Epoch 2/25: 100%|██████████| 112/112 [01:48<00:00,  1.03it/s]


Epoch 2/25: Train Loss=0.9926, Val Loss=1.2476, Train Acc=0.7085, Val Acc=0.7008, Val macro-F1=0.3284


Epoch 3/25: 100%|██████████| 112/112 [01:50<00:00,  1.01it/s]


Epoch 3/25: Train Loss=0.9252, Val Loss=1.2358, Train Acc=0.7464, Val Acc=0.7165, Val macro-F1=0.3128


... (26 outputs left out)

Epoch 17/25: 100%|██████████| 112/112 [01:50<00:00,  1.01it/s]


Epoch 17/25: Train Loss=0.6173, Val Loss=1.3190, Train Acc=0.9371, Val Acc=0.7028, Val macro-F1=0.3389


Epoch 18/25: 100%|██████████| 112/112 [01:54<00:00,  1.02s/it]


Epoch 18/25: Train Loss=0.6220, Val Loss=1.3210, Train Acc=0.9340, Val Acc=0.6949, Val macro-F1=0.3365
Early stopping at epoch 18
Best epoch: 10 (val macro-F1 = 0.3499)


In [130]:
# Save the trained VGG16 age-group model weights.
save_path = MODEL_DIR / "vgg16_age_group.pth"
torch.save(vgg_model.state_dict(), save_path)
print(f"Saved VGG16 age-group model to: {save_path}")


Saved VGG16 age-group model to: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\age_group\vgg16_age_group.pth


### EfficientNet-B0:

In [131]:
effnet_model = effnet_model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)   # CHANGED
optimizer = optim.AdamW(
    [
        {"params": effnet_model.features[5:].parameters(), "lr": 1e-5},   # CHANGED
        {"params": effnet_model.classifier.parameters(),   "lr": 1e-3},
    ],
    weight_decay=0.05
)

effnet_history = train_model(effnet_model, optimizer, criterion, num_epochs=25)   # CHANGED

effnet_model = effnet_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Epoch 1/25: 100%|██████████| 112/112 [01:33<00:00,  1.20it/s]


Epoch 1/25: Train Loss=1.1822, Val Loss=1.0389, Train Acc=0.6181, Val Acc=0.7047, Val macro-F1=0.3758


Epoch 2/25: 100%|██████████| 112/112 [01:31<00:00,  1.22it/s]


Epoch 2/25: Train Loss=1.0706, Val Loss=1.1346, Train Acc=0.6580, Val Acc=0.6850, Val macro-F1=0.3657


Epoch 3/25: 100%|██████████| 112/112 [01:33<00:00,  1.20it/s]


Epoch 3/25: Train Loss=1.0156, Val Loss=1.2130, Train Acc=0.6799, Val Acc=0.6496, Val macro-F1=0.3493


... (30 outputs left out)

Epoch 19/25: 100%|██████████| 112/112 [01:30<00:00,  1.24it/s]


Epoch 19/25: Train Loss=0.8337, Val Loss=1.1900, Train Acc=0.7911, Val Acc=0.6949, Val macro-F1=0.3718


Epoch 20/25: 100%|██████████| 112/112 [01:27<00:00,  1.28it/s]


Epoch 20/25: Train Loss=0.8137, Val Loss=1.1690, Train Acc=0.8076, Val Acc=0.6850, Val macro-F1=0.3570
Early stopping at epoch 20
Best epoch: 12 (val macro-F1 = 0.3835)


In [132]:
# Save the trained EfficientNet-B0 age-group model weights.
save_path = MODEL_DIR / "efficientnet_b0_age_group.pth"
torch.save(effnet_model.state_dict(), save_path)
print(f"Saved EfficientNet-B0 age-group model to: {save_path}")


Saved EfficientNet-B0 age-group model to: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\age_group\efficientnet_b0_age_group.pth


### ViT:

In [133]:
vit_model = vit_model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)   # CHANGED
optimizer = optim.AdamW(
    [
        {"params": [p for l in vit_model.encoder.layers[-3:] for p in l.parameters()], "lr": 1e-5},   # CHANGED
        {"params": vit_model.encoder.ln.parameters(), "lr": 1e-5},
        {"params": vit_model.heads.parameters(),      "lr": 1e-3},
    ],
    weight_decay=0.05
)

vit_history = train_model(vit_model, optimizer, criterion, num_epochs=25)   # CHANGED

vit_model = vit_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()



Epoch 1/25: 100%|██████████| 112/112 [03:25<00:00,  1.83s/it]


Epoch 1/25: Train Loss=1.0484, Val Loss=1.1283, Train Acc=0.6804, Val Acc=0.7008, Val macro-F1=0.3397


Epoch 2/25: 100%|██████████| 112/112 [03:23<00:00,  1.82s/it]


Epoch 2/25: Train Loss=0.8991, Val Loss=1.1923, Train Acc=0.7515, Val Acc=0.6850, Val macro-F1=0.3679


Epoch 3/25: 100%|██████████| 112/112 [03:23<00:00,  1.81s/it]


Epoch 3/25: Train Loss=0.8120, Val Loss=1.1875, Train Acc=0.7953, Val Acc=0.7028, Val macro-F1=0.3526


... (24 outputs left out)

Epoch 16/25: 100%|██████████| 112/112 [03:38<00:00,  1.95s/it]


Epoch 16/25: Train Loss=0.5086, Val Loss=1.2611, Train Acc=0.9837, Val Acc=0.7126, Val macro-F1=0.3684


Epoch 17/25: 100%|██████████| 112/112 [03:28<00:00,  1.86s/it]


Epoch 17/25: Train Loss=0.4995, Val Loss=1.2587, Train Acc=0.9876, Val Acc=0.7028, Val macro-F1=0.3633
Early stopping at epoch 17
Best epoch: 9 (val macro-F1 = 0.3885)


In [134]:
# Save the trained ViT-B/16 age-group model weights.
save_path = MODEL_DIR / "vit_b16_age_group.pth"
torch.save(vit_model.state_dict(), save_path)
print(f"Saved ViT-B/16 age-group model to: {save_path}")


Saved ViT-B/16 age-group model to: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\age_group\vit_b16_age_group.pth


### Training curves (overfitting check):

In [135]:
histories = {}
for name, var in [("CNN", "cnn"), ("ResNet50", "resnet"), ("VGG16", "vgg"), ("EfficientNet_B0", "effnet"), ("ViT_B16", "vit")]:
    if var + "_history" in globals():
        histories[name] = globals()[var + "_history"]

fig = go.Figure()
for name, h in histories.items():
    fig.add_scatter(y=h["train_loss"], mode="lines", name=name + " train")
    fig.add_scatter(y=h["val_loss"], mode="lines", line=dict(dash="dash"), name=name + " val")
fig.update_layout(height=450, width=850, title="Loss per Epoch (solid = train, dashed = validation)",
                  xaxis_title="Epoch", yaxis_title="Loss")
fig.show()

fig = go.Figure()
for name, h in histories.items():
    fig.add_scatter(y=h["val_f1"], mode="lines", name=name)
fig.update_layout(height=450, width=850, title="Validation macro-F1 per Epoch",
                  xaxis_title="Epoch", yaxis_title="Macro-F1")
fig.show()

fig = go.Figure()
for name, h in histories.items():
    fig.add_scatter(y=h["val_acc"], mode="lines", name=name)
fig.update_layout(height=450, width=850, title="Validation Accuracy per Epoch",
                  xaxis_title="Epoch", yaxis_title="Accuracy")
fig.show()

# Model Evaluation:

In [136]:
def evaluate_model(model, loader, dataset_name="Set", show_report=False):
    model = model.to(device)
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(probs.argmax(dim=1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    model = model.to("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    all_labels = np.asarray(all_labels, dtype=int)
    all_preds = np.asarray(all_preds, dtype=int)
    all_probs = np.asarray(all_probs)

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, labels=np.arange(num_classes), average="macro", zero_division=0)
    rec = recall_score(all_labels, all_preds, labels=np.arange(num_classes), average="macro", zero_division=0)
    f1 = f1_score(all_labels, all_preds, labels=np.arange(num_classes), average="macro", zero_division=0)

    try:
        auc = roc_auc_score(all_labels, all_probs, labels=np.arange(num_classes),
                            multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")
        print(f"Warning: AUC could not be calculated for {dataset_name}.")

    # CHANGED: within-one-bin accuracy over the ordered age classes (indices 0-3 only).
    ordered = (all_labels <= 3) & (all_preds <= 3)
    ordered_true = (all_labels <= 3)
    w1 = (np.abs(all_labels[ordered] - all_preds[ordered]) <= 1).sum() / max(ordered_true.sum(), 1)

    print(f"--- {dataset_name} Metrics ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC-ROC: {auc:.4f}")
    print(f"Within-one-bin accuracy (age bins 18-25..50+): {w1:.4f}\n")

    if show_report:
        print(classification_report(all_labels, all_preds, labels=np.arange(num_classes),
                                    target_names=class_names, zero_division=0))

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auc": auc,
            "within_one_bin": w1, "labels": all_labels, "preds": all_preds, "probs": all_probs}

### CNN Model Result:

In [137]:
print("===== CNN RESULTS =====\n")
cnn_train_results = evaluate_model(cnn_model, train_eval_loader, "Train")
cnn_val_results = evaluate_model(cnn_model, val_loader, "Validation")
cnn_test_results = evaluate_model(cnn_model, test_loader, "Test", show_report=True)

===== CNN RESULTS =====

--- Train Metrics ---
Accuracy:  0.7891
Precision: 0.4749
Recall:    0.3650
F1-Score:  0.3491
AUC-ROC: 0.8744
Within-one-bin accuracy (age bins 18-25..50+): 0.9113

--- Validation Metrics ---
Accuracy:  0.6969
Precision: 0.3729
Recall:    0.3277
F1-Score:  0.2791
AUC-ROC: 0.8167
Within-one-bin accuracy (age bins 18-25..50+): 0.7061

--- Test Metrics ---
Accuracy:  0.7615
Precision: 0.2748
Recall:    0.3221
F1-Score:  0.2875
AUC-ROC: 0.8263
Within-one-bin accuracy (age bins 18-25..50+): 0.8568

                precision    recall  f1-score   support

         18-25       0.00      0.00      0.00        59
         26-35       0.56      0.96      0.70       288
         36-50       0.00      0.00      0.00        76
           50+       0.10      0.04      0.05        52
not_applicable       1.00      0.94      0.97       532
      under_18       0.00      0.00      0.00        12

      accuracy                           0.76      1019
     macro avg       0.27 

### ResNet 50 Model Result:

In [138]:
print("===== RESNET50 RESULTS =====\n")
resnet_train_results = evaluate_model(resnet_model, train_eval_loader, "Train")
resnet_val_results = evaluate_model(resnet_model, val_loader, "Validation")
resnet_test_results = evaluate_model(resnet_model, test_loader, "Test", show_report=True)

===== RESNET50 RESULTS =====

--- Train Metrics ---
Accuracy:  0.8099
Precision: 0.6207
Recall:    0.7327
F1-Score:  0.6584
AUC-ROC: 0.9557
Within-one-bin accuracy (age bins 18-25..50+): 0.9275

--- Validation Metrics ---
Accuracy:  0.6929
Precision: 0.4554
Recall:    0.4457
F1-Score:  0.4202
AUC-ROC: 0.8577
Within-one-bin accuracy (age bins 18-25..50+): 0.7149

--- Test Metrics ---
Accuracy:  0.7527
Precision: 0.5054
Recall:    0.4920
F1-Score:  0.4840
AUC-ROC: 0.8986
Within-one-bin accuracy (age bins 18-25..50+): 0.8547

                precision    recall  f1-score   support

         18-25       0.27      0.37      0.31        59
         26-35       0.64      0.61      0.62       288
         36-50       0.42      0.42      0.42        76
           50+       0.63      0.33      0.43        52
not_applicable       0.98      0.97      0.98       532
      under_18       0.10      0.25      0.15        12

      accuracy                           0.75      1019
     macro avg       

### VGG16 Model Result:

In [139]:
print("===== VGG16 RESULTS =====\n")
vgg_train_results = evaluate_model(vgg_model, train_eval_loader, "Train")
vgg_val_results = evaluate_model(vgg_model, val_loader, "Validation")
vgg_test_results = evaluate_model(vgg_model, test_loader, "Test", show_report=True)

===== VGG16 RESULTS =====

--- Train Metrics ---
Accuracy:  0.9705
Precision: 0.9510
Recall:    0.9572
F1-Score:  0.9539
AUC-ROC: 0.9981
Within-one-bin accuracy (age bins 18-25..50+): 0.9964

--- Validation Metrics ---
Accuracy:  0.7047
Precision: 0.4499
Recall:    0.3694
F1-Score:  0.3499
AUC-ROC: 0.7325
Within-one-bin accuracy (age bins 18-25..50+): 0.7456

--- Test Metrics ---
Accuracy:  0.7851
Precision: 0.5009
Recall:    0.4269
F1-Score:  0.4365
AUC-ROC: 0.7724
Within-one-bin accuracy (age bins 18-25..50+): 0.9011

                precision    recall  f1-score   support

         18-25       0.27      0.32      0.29        59
         26-35       0.63      0.84      0.72       288
         36-50       0.35      0.14      0.21        76
           50+       0.75      0.29      0.42        52
not_applicable       1.00      0.96      0.98       532
      under_18       0.00      0.00      0.00        12

      accuracy                           0.79      1019
     macro avg       0.5

### EfficientNet-B0 Model Result:

In [140]:
print("===== EFFICIENTNET_B0 RESULTS =====\n")
effnet_train_results = evaluate_model(effnet_model, train_eval_loader, "Train")
effnet_val_results = evaluate_model(effnet_model, val_loader, "Validation")
effnet_test_results = evaluate_model(effnet_model, test_loader, "Test", show_report=True)

===== EFFICIENTNET_B0 RESULTS =====

--- Train Metrics ---
Accuracy:  0.8896
Precision: 0.7753
Recall:    0.8362
F1-Score:  0.8002
AUC-ROC: 0.9806
Within-one-bin accuracy (age bins 18-25..50+): 0.9718

--- Validation Metrics ---
Accuracy:  0.6969
Precision: 0.4463
Recall:    0.3943
F1-Score:  0.3835
AUC-ROC: 0.8007
Within-one-bin accuracy (age bins 18-25..50+): 0.7368

--- Test Metrics ---
Accuracy:  0.7645
Precision: 0.5107
Recall:    0.4461
F1-Score:  0.4614
AUC-ROC: 0.8471
Within-one-bin accuracy (age bins 18-25..50+): 0.9011

                precision    recall  f1-score   support

         18-25       0.21      0.25      0.23        59
         26-35       0.62      0.76      0.68       288
         36-50       0.37      0.25      0.30        76
           50+       0.68      0.29      0.41        52
not_applicable       1.00      0.96      0.98       532
      under_18       0.18      0.17      0.17        12

      accuracy                           0.76      1019
     macro avg

### ViT Model Result:

In [141]:
print("===== VIT_B16 RESULTS =====\n")
vit_train_results = evaluate_model(vit_model, train_eval_loader, "Train")
vit_val_results = evaluate_model(vit_model, val_loader, "Validation")
vit_test_results = evaluate_model(vit_model, test_loader, "Test", show_report=True)

===== VIT_B16 RESULTS =====

--- Train Metrics ---
Accuracy:  0.9739
Precision: 0.9533
Recall:    0.9649
F1-Score:  0.9587
AUC-ROC: 0.9988
Within-one-bin accuracy (age bins 18-25..50+): 0.9970

--- Validation Metrics ---
Accuracy:  0.7224
Precision: 0.5943
Recall:    0.4059
F1-Score:  0.3885
AUC-ROC: 0.7419
Within-one-bin accuracy (age bins 18-25..50+): 0.7500

--- Test Metrics ---
Accuracy:  0.7723
Precision: 0.6619
Recall:    0.4303
F1-Score:  0.4591
AUC-ROC: 0.7738
Within-one-bin accuracy (age bins 18-25..50+): 0.9032

                precision    recall  f1-score   support

         18-25       0.28      0.19      0.22        59
         26-35       0.60      0.79      0.68       288
         36-50       0.30      0.25      0.27        76
           50+       0.80      0.31      0.44        52
not_applicable       1.00      0.96      0.98       532
      under_18       1.00      0.08      0.15        12

      accuracy                           0.77      1019
     macro avg       0

### Result Summary:

In [142]:
def create_results_summary(train_results, val_results, test_results):

    return pd.DataFrame({
        "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"],
        "Train": [train_results["accuracy"], train_results["precision"], train_results["recall"],
                  train_results["f1"], train_results["auc"]],
        "Validation": [val_results["accuracy"], val_results["precision"], val_results["recall"],
                       val_results["f1"], val_results["auc"]],
        "Test": [test_results["accuracy"], test_results["precision"], test_results["recall"],
                 test_results["f1"], test_results["auc"]]
    })

In [143]:
model_list = [("CNN", "cnn"), ("ResNet50", "resnet"), ("VGG16", "vgg"), ("EfficientNet_B0", "effnet"), ("ViT_B16", "vit")]

for name, var in model_list:
    if var + "_test_results" in globals():
        print(f"\n===== {name.upper()} RESULTS =====\n")
        print(create_results_summary(globals()[var + "_train_results"],
                                     globals()[var + "_val_results"],
                                     globals()[var + "_test_results"]))



# Compact comparison on the test set (includes within-one-bin accuracy)
rows = []
for name, var in model_list:
    if var + "_test_results" in globals():
        r = globals()[var + "_test_results"]
        rows.append({"Model": name, "Accuracy": r["accuracy"], "Macro-F1": r["f1"],
                     "AUC-ROC": r["auc"], "Within-1-bin": r["within_one_bin"]})
print("\n===== TEST SET COMPARISON =====\n")
print(pd.DataFrame(rows).round(4).to_string(index=False))


===== CNN RESULTS =====

      Metric     Train  Validation      Test
0   Accuracy  0.789104    0.696850  0.761531
1  Precision  0.474909    0.372935  0.274802
2     Recall  0.364977    0.327684  0.322148
3   F1-Score  0.349078    0.279081  0.287482
4    AUC-ROC  0.874353    0.816683  0.826260

===== RESNET50 RESULTS =====

      Metric     Train  Validation      Test
0   Accuracy  0.809885    0.692913  0.752699
1  Precision  0.620653    0.455445  0.505351
2     Recall  0.732698    0.445675  0.492030
3   F1-Score  0.658441    0.420221  0.484047
4    AUC-ROC  0.955668    0.857660  0.898624

===== VGG16 RESULTS =====

      Metric     Train  Validation      Test
0   Accuracy  0.970514    0.704724  0.785083
1  Precision  0.950973    0.449864  0.500865
2     Recall  0.957238    0.369406  0.426898
3   F1-Score  0.953942    0.349876  0.436505
4    AUC-ROC  0.998149    0.732532  0.772432

===== EFFICIENTNET_B0 RESULTS =====

      Metric     Train  Validation      Test
0   Accuracy  0.889638

# Confusion Matrices

All confusion matrices use the six age-group labels in `class_names` order. Rows represent actual classes and columns represent predicted classes.


### CNN Model:

In [144]:
# Train Confusion Matrix
cm_cnn_train = confusion_matrix(
                        cnn_train_results["labels"],
                        cnn_train_results["preds"]
                    )

cnn_train_fig = px.imshow(
                        cm_cnn_train,
                        text_auto=True,
                        x=class_names,    
                        y=class_names,
                        color_continuous_scale='Blues',
                        title='Confusion Matrix for CNN Model - Train'
                    )

cnn_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
cnn_train_fig.show()
joblib.dump(cnn_train_fig, CHARTS_DIR / "confusion_matrix_age_group_cnn_train.pkl")

# Validation Confusion Matrix
cm_cnn_val = confusion_matrix(
                        cnn_val_results["labels"],
                        cnn_val_results["preds"]
                    )

cnn_val_fig = px.imshow(
                        cm_cnn_val,
                        text_auto=True,
                        x=class_names,    
                        y=class_names,
                        color_continuous_scale='Blues',
                        title='Confusion Matrix for CNN Model - Validation'
                    )

cnn_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
cnn_val_fig.show()
joblib.dump(cnn_val_fig, CHARTS_DIR / "confusion_matrix_age_group_cnn_val.pkl")

# Test Confusion Matrix
cm_cnn_test = confusion_matrix(
                        cnn_test_results["labels"],
                        cnn_test_results["preds"]
                    )

cnn_test_fig = px.imshow(
                        cm_cnn_test,
                        text_auto=True,
                        x=class_names,    
                        y=class_names,
                        color_continuous_scale='Blues',
                        title='Confusion Matrix for CNN Model - Test'
                    )

cnn_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
cnn_test_fig.show()
joblib.dump(cnn_test_fig, CHARTS_DIR / "confusion_matrix_age_group_cnn_test.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\age_group\\confusion_matrix_age_group_cnn_test.pkl']

### ResNet50 Model:

In [145]:
# Train Confusion Matrix
cm_resnet_train = confusion_matrix(
                        resnet_train_results["labels"],
                        resnet_train_results["preds"]
                    )

resnet_train_fig = px.imshow(
                        cm_resnet_train,
                        text_auto=True,
                        x=class_names,    
                        y=class_names,
                        color_continuous_scale='Greens',
                        title='Confusion Matrix for ResNet50 Model - Train'
                    )

resnet_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
resnet_train_fig.show()
joblib.dump(resnet_train_fig, CHARTS_DIR / "confusion_matrix_age_group_resnet50_train.pkl")

# Validation Confusion Matrix
cm_resnet_val = confusion_matrix(
                        resnet_val_results["labels"],
                        resnet_val_results["preds"]
                    )

resnet_val_fig = px.imshow(
                        cm_resnet_val,
                        text_auto=True,
                        x=class_names,    
                        y=class_names,
                        color_continuous_scale='Greens',
                        title='Confusion Matrix for ResNet50 Model - Validation'
                    )

resnet_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
resnet_val_fig.show()
joblib.dump(resnet_val_fig, CHARTS_DIR / "confusion_matrix_age_group_resnet50_val.pkl")

# Test Confusion Matrix
cm_resnet_test = confusion_matrix(
                        resnet_test_results["labels"],
                        resnet_test_results["preds"]
                    )

resnet_test_fig = px.imshow(
                        cm_resnet_test,
                        text_auto=True,
                        x=class_names,    
                        y=class_names,
                        color_continuous_scale='Greens',
                        title='Confusion Matrix for ResNet50 Model - Test'
                    )

resnet_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
resnet_test_fig.show()
joblib.dump(resnet_test_fig, CHARTS_DIR / "confusion_matrix_age_group_resnet50_test.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\age_group\\confusion_matrix_age_group_resnet50_test.pkl']

### VGG16 Model:

In [146]:
# Train Confusion Matrix
cm_vgg_train = confusion_matrix(
                        vgg_train_results["labels"],
                        vgg_train_results["preds"]
                    )

vgg_train_fig = px.imshow(
                        cm_vgg_train,
                        text_auto=True,
                        x=class_names,    
                        y=class_names,
                        color_continuous_scale='Reds',
                        title='Confusion Matrix for VGG16 Model - Train'
                    )

vgg_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vgg_train_fig.show()
joblib.dump(vgg_train_fig, CHARTS_DIR / "confusion_matrix_age_group_Vgg_16_train.pkl")

# Validation Confusion Matrix
cm_vgg_val = confusion_matrix(
                        vgg_val_results["labels"],
                        vgg_val_results["preds"]
                    )

vgg_val_fig = px.imshow(
                        cm_vgg_val,
                        text_auto=True,
                        x=class_names,    
                        y=class_names,
                        color_continuous_scale='Reds',
                        title='Confusion Matrix for VGG16 Model - Validation'
                    )

vgg_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vgg_val_fig.show()
joblib.dump(vgg_val_fig, CHARTS_DIR / "confusion_matrix_age_group_Vgg_16_val.pkl")

# Test Confusion Matrix
cm_vgg_test = confusion_matrix(
                        vgg_test_results["labels"],
                        vgg_test_results["preds"]
                    )

vgg_test_fig = px.imshow(
                        cm_vgg_test,
                        text_auto=True,
                        x=class_names,
                        y=class_names,
                        color_continuous_scale='Reds',
                        title='Confusion Matrix for VGG16 Model - Test'
                    )

vgg_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vgg_test_fig.show()
joblib.dump(vgg_test_fig, CHARTS_DIR / "confusion_matrix_age_group_Vgg_16_test.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\age_group\\confusion_matrix_age_group_Vgg_16_test.pkl']

### EfficientNet_B0 Model:

In [147]:
# Train Confusion Matrix
cm_effnet_train = confusion_matrix(
                        effnet_train_results["labels"],
                        effnet_train_results["preds"]
                    )

effnet_train_fig = px.imshow(
                        cm_effnet_train,
                        text_auto=True,
                        x=class_names,
                        y=class_names,
                        color_continuous_scale='Purples',
                        title='Confusion Matrix for EfficientNet_B0 Model - Train'
                    )

effnet_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
effnet_train_fig.show()
joblib.dump(effnet_train_fig, CHARTS_DIR / "confusion_matrix_age_group_EffNet_B0_train.pkl")

# Validation Confusion Matrix
cm_effnet_val = confusion_matrix(
                        effnet_val_results["labels"],
                        effnet_val_results["preds"]
                    )

effnet_val_fig = px.imshow(
                        cm_effnet_val,
                        text_auto=True,
                        x=class_names,
                        y=class_names,
                        color_continuous_scale='Purples',
                        title='Confusion Matrix for EfficientNet_B0 Model - Validation'
                    )

effnet_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
effnet_val_fig.show()
joblib.dump(effnet_val_fig, CHARTS_DIR / "confusion_matrix_age_group_EffNet_B0_val.pkl")

# Test Confusion Matrix
cm_effnet_test = confusion_matrix(
                        effnet_test_results["labels"],
                        effnet_test_results["preds"]
                    )

effnet_test_fig = px.imshow(
                        cm_effnet_test,
                        text_auto=True,
                        x=class_names,
                        y=class_names,
                        color_continuous_scale='Purples',
                        title='Confusion Matrix for EfficientNet_B0 Model - Test'
                    )

effnet_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
effnet_test_fig.show()
joblib.dump(effnet_test_fig, CHARTS_DIR / "confusion_matrix_age_group_EffNet_B0_test.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\age_group\\confusion_matrix_age_group_EffNet_B0_test.pkl']

### ViT_B16 Model:

In [148]:
# Train Confusion Matrix
cm_vit_train = confusion_matrix(
                        vit_train_results["labels"],
                        vit_train_results["preds"]
                    )

vit_train_fig = px.imshow(
                        cm_vit_train,
                        text_auto=True,
                        x=class_names,
                        y=class_names,
                        color_continuous_scale='Oranges',
                        title='Confusion Matrix for ViT_B16 Model - Train'
                    )

vit_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vit_train_fig.show()
joblib.dump(vit_train_fig, CHARTS_DIR / "confusion_matrix_age_group_ViT_B16_train.pkl")

# Validation Confusion Matrix
cm_vit_val = confusion_matrix(
                        vit_val_results["labels"],
                        vit_val_results["preds"]
                    )

vit_val_fig = px.imshow(
                        cm_vit_val,
                        text_auto=True,
                        x=class_names,
                        y=class_names,
                        color_continuous_scale='Oranges',
                        title='Confusion Matrix for ViT_B16 Model - Validation'
                    )

vit_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vit_val_fig.show()
joblib.dump(vit_val_fig, CHARTS_DIR / "confusion_matrix_age_group_ViT_B16_val.pkl")

# Test Confusion Matrix
cm_vit_test = confusion_matrix(
                        vit_test_results["labels"],
                        vit_test_results["preds"]
                    )

vit_test_fig = px.imshow(
                        cm_vit_test,
                        text_auto=True,
                        x=class_names,
                        y=class_names,
                        color_continuous_scale='Oranges',
                        title='Confusion Matrix for ViT_B16 Model - Test'
                    )

vit_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vit_test_fig.show()
joblib.dump(vit_test_fig, CHARTS_DIR / "confusion_matrix_age_group_ViT_B16_test.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\age_group\\confusion_matrix_age_group_ViT_B16_test.pkl']

# ROC Curve


### CNN - ROC Curve:

In [174]:
# Train ROC
fpr_tr, tpr_tr, _ = roc_curve(
    np.eye(num_classes)[cnn_train_results["labels"]].ravel(),
    cnn_train_results["probs"].ravel()
)

# Validation ROC
fpr_va, tpr_va, _ = roc_curve(
    np.eye(num_classes)[cnn_val_results["labels"]].ravel(),
    cnn_val_results["probs"].ravel()
)

# Test ROC
fpr_te, tpr_te, _ = roc_curve(
    np.eye(num_classes)[cnn_test_results["labels"]].ravel(),
    cnn_test_results["probs"].ravel()
)

fig = go.Figure()

fig.add_scatter(
    x=fpr_tr,
    y=tpr_tr,
    mode='lines',
    name=f'Train (AUC={auc(fpr_tr, tpr_tr):.3f})'
)

fig.add_scatter(
    x=fpr_va,
    y=tpr_va,
    mode='lines',
    name=f'Validation (AUC={auc(fpr_va, tpr_va):.3f})'
)

fig.add_scatter(
    x=fpr_te,
    y=tpr_te,
    mode='lines',
    name=f'Test (AUC={auc(fpr_te, tpr_te):.3f})'
)

fig.add_scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    line=dict(dash='dash'),
    name='Random'
)

fig.update_layout(
    height=500, width=800,
    title='ROC Curve - Custom CNN',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate'
)

fig.show()

joblib.dump(fig, CHARTS_DIR / "ROC_Curve_Age_group-Custom_CNN_pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\age_group\\ROC_Curve_Age_group-Custom_CNN_pkl']

### ResNet50 - ROC Curve:

In [167]:
# Train ROC
fpr_tr, tpr_tr, _ = roc_curve(
    np.eye(num_classes)[resnet_train_results["labels"]].ravel(),
    resnet_train_results["probs"].ravel()
)

# Validation ROC
fpr_va, tpr_va, _ = roc_curve(
    np.eye(num_classes)[resnet_val_results["labels"]].ravel(),
    resnet_val_results["probs"].ravel()
)

# Test ROC
fpr_te, tpr_te, _ = roc_curve(
    np.eye(num_classes)[resnet_test_results["labels"]].ravel(),
    resnet_test_results["probs"].ravel()
)

fig = go.Figure()

fig.add_scatter(
    x=fpr_tr,
    y=tpr_tr,
    mode='lines',
    name=f'Train (AUC={auc(fpr_tr, tpr_tr):.3f})'
)

fig.add_scatter(
    x=fpr_va,
    y=tpr_va,
    mode='lines',
    name=f'Validation (AUC={auc(fpr_va, tpr_va):.3f})'
)

fig.add_scatter(
    x=fpr_te,
    y=tpr_te,
    mode='lines',
    name=f'Test (AUC={auc(fpr_te, tpr_te):.3f})'
)

fig.add_scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    line=dict(dash='dash'),
    name='Random'
)

fig.update_layout(
    height=500, width=800,
    title='ROC Curve - ResNet50',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate'
)

fig.show()

joblib.dump(fig, CHARTS_DIR / "ROC_Curve_Age_group-ResNet50.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\age_group\\ROC_Curve_Age_group-ResNet50.pkl']

### VGG-B16:

In [168]:
# Train ROC
fpr_tr, tpr_tr, _ = roc_curve(
    np.eye(num_classes)[vgg_train_results["labels"]].ravel(),
    vgg_train_results["probs"].ravel()
)

# Validation ROC
fpr_va, tpr_va, _ = roc_curve(
    np.eye(num_classes)[vgg_val_results["labels"]].ravel(),
    vgg_val_results["probs"].ravel()
)

# Test ROC
fpr_te, tpr_te, _ = roc_curve(
    np.eye(num_classes)[vgg_test_results["labels"]].ravel(),
    vgg_test_results["probs"].ravel()
)

fig = go.Figure()

fig.add_scatter(
    x=fpr_tr,
    y=tpr_tr,
    mode='lines',
    name=f'Train (AUC={auc(fpr_tr, tpr_tr):.3f})'
)

fig.add_scatter(
    x=fpr_va,
    y=tpr_va,
    mode='lines',
    name=f'Validation (AUC={auc(fpr_va, tpr_va):.3f})'
)

fig.add_scatter(
    x=fpr_te,
    y=tpr_te,
    mode='lines',
    name=f'Test (AUC={auc(fpr_te, tpr_te):.3f})'
)

fig.add_scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    line=dict(dash='dash'),
    name='Random'
)

fig.update_layout(
    height=500, width=800,
    title='ROC Curve - VGG16',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate'
)

fig.show()

joblib.dump(fig, CHARTS_DIR / "ROC_Curve_Age_group-VGG16.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\age_group\\ROC_Curve_Age_group-VGG16.pkl']

### EffNet - ROC Curve

In [165]:
# Train ROC
fpr_tr, tpr_tr, _ = roc_curve(
    np.eye(num_classes)[effnet_train_results["labels"]].ravel(),
    effnet_train_results["probs"].ravel()
)

# Validation ROC
fpr_va, tpr_va, _ = roc_curve(
    np.eye(num_classes)[effnet_val_results["labels"]].ravel(),
    effnet_val_results["probs"].ravel()
)

# Test ROC
fpr_te, tpr_te, _ = roc_curve(
    np.eye(num_classes)[effnet_test_results["labels"]].ravel(),
    effnet_test_results["probs"].ravel()
)

fig = go.Figure()

fig.add_scatter(
    x=fpr_tr,
    y=tpr_tr,
    mode='lines',
    name=f'Train (AUC={auc(fpr_tr, tpr_tr):.3f})'
)

fig.add_scatter(
    x=fpr_va,
    y=tpr_va,
    mode='lines',
    name=f'Validation (AUC={auc(fpr_va, tpr_va):.3f})'
)

fig.add_scatter(
    x=fpr_te,
    y=tpr_te,
    mode='lines',
    name=f'Test (AUC={auc(fpr_te, tpr_te):.3f})'
)

fig.add_scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    line=dict(dash='dash'),
    name='Random'
)

fig.update_layout(
    height=500, width=800,
    title='ROC Curve - EfficientNet-B0',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate'
)

fig.show()

joblib.dump(fig, CHARTS_DIR / "ROC_Curve_Age_group-EfficientNet_B0.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\age_group\\ROC_Curve_Age_group-EfficientNet_B0.pkl']

### ViT :

In [171]:
# Train ROC
fpr_tr, tpr_tr, _ = roc_curve(
    np.eye(num_classes)[vit_train_results["labels"]].ravel(),
    vit_train_results["probs"].ravel()
)

# Validation ROC
fpr_va, tpr_va, _ = roc_curve(
    np.eye(num_classes)[vit_val_results["labels"]].ravel(),
    vit_val_results["probs"].ravel()
)

# Test ROC
fpr_te, tpr_te, _ = roc_curve(
    np.eye(num_classes)[vit_test_results["labels"]].ravel(),
    vit_test_results["probs"].ravel()
)

fig = go.Figure()

fig.add_scatter(
    x=fpr_tr,
    y=tpr_tr,
    mode='lines',
    name=f'Train (AUC={auc(fpr_tr, tpr_tr):.3f})'
)

fig.add_scatter(
    x=fpr_va,
    y=tpr_va,
    mode='lines',
    name=f'Validation (AUC={auc(fpr_va, tpr_va):.3f})'
)

fig.add_scatter(
    x=fpr_te,
    y=tpr_te,
    mode='lines',
    name=f'Test (AUC={auc(fpr_te, tpr_te):.3f})'
)

fig.add_scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    line=dict(dash='dash'),
    name='Random'
)

fig.update_layout(
    height=500, width=800,
    title='ROC Curve - ViT-B/16',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate'
)

fig.show()

joblib.dump(fig, CHARTS_DIR / "ROC_Curve_Age_group-ViT-B16.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\age_group\\ROC_Curve_Age_group-ViT-B16.pkl']

# Compare the models

The best model is selected using validation macro F1-score, not test performance. This keeps the test set reserved for final evaluation.

In [175]:
rows = []
for name, var in model_list:
    if var + "_test_results" in globals():
        val_r = globals()[var + "_val_results"]
        test_r = globals()[var + "_test_results"]
        rows.append({"Model": name,
                     "Val F1": round(val_r["f1"], 4),
                     "Test Accuracy": round(test_r["accuracy"], 4),
                     "Test F1": round(test_r["f1"], 4),
                     "Test AUC": round(test_r["auc"], 4)})

comparison = pd.DataFrame(rows).sort_values("Val F1", ascending=False).reset_index(drop=True)
comparison.to_csv(MODEL_DIR / "results.csv", index=False)
best_model_name = comparison.loc[0, "Model"]
print("Best model for", TASK, ":", best_model_name)
comparison

Best model for age : ResNet50


,Model,Val F1,Test Accuracy,Test F1,Test AUC
0,ResNet50,0.4202,0.7527,0.4840,0.8986
1,ViT_B16,0.3885,0.7723,0.4591,0.7738
2,EfficientNet_B0,0.3835,0.7645,0.4614,0.8471
3,VGG16,0.3499,0.7851,0.4365,0.7724
4,CNN,0.2791,0.7615,0.2875,0.8263
